# Numbers, Bits, Truth, and Precision
Numbers feel friendly until precision, representation, and truthiness begin to disagree with intuition. The goal here is to connect the Python-level experience of numbers with the lower-level rules that make those surprises happen.

When people struggle with **Numbers, Bits, Truth, and Precision**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


- Understand why floats behave the way they do
- See truthiness as a runtime rule set
- Connect operator results to underlying representation
- Use safer numeric comparisons


Numbers are still objects in Python. Small integers may be reused internally, floats hold approximate binary data, and booleans are singletons used heavily in control flow. The important human lesson is that mathematical intent and machine representation are not always identical.


In [1]:
a = 256
b = 256
print(id(a), id(b))
print(a is b)
print(id(True), id(False))


140715746456328 140715746456328
True
140715744926568 140715744926600


Bytecode helps show that conditions are just operations producing values that the interpreter then tests. It also shows that literal constants are loaded as objects.


In [2]:
import dis

def compare(x, y):
    if x < y:
        return True
    return False

dis.dis(compare)


  3           0 RESUME                   0

  4           2 LOAD_FAST                0 (x)
              4 LOAD_FAST                1 (y)
              6 COMPARE_OP               0 (<)
             12 POP_JUMP_FORWARD_IF_FALSE     2 (to 18)

  5          14 LOAD_CONST               1 (True)
             16 RETURN_VALUE

  6     >>   18 LOAD_CONST               2 (False)
             20 RETURN_VALUE


They are not raw machine words from your point of view, even though the implementation matters underneath.

Binary floating-point simply cannot represent every decimal exactly.

Many objects can be treated as true or false in conditions.

Learning those rules once is better than being surprised repeatedly.


This is the famous example, but the real lesson is not the joke. The lesson is representation.


In [3]:
import math
print(0.1 + 0.2)
print((0.1 + 0.2) == 0.3)
print(math.isclose(0.1 + 0.2, 0.3))


0.30000000000000004
False
True


Python does not require every condition to be a literal True or False.


In [4]:
values = [0, 1, "", "text", [], [1], None, {"a": 1}]
print([(repr(v), bool(v)) for v in values])


[('0', False), ('1', True), ("''", False), ("'text'", True), ('[]', False), ('[1]', True), ('None', False), ("{'a': 1}", True)]


This is deeply Pythonic and often forgotten.


In [5]:
print([] or "fallback")
print("hello" and 42)


fallback
42


This is not only a classroom topic. **Numbers, Bits, Truth, and Precision** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


- Comparing floats directly when tolerance would be safer
- Assuming `and` and `or` always return booleans
- Forgetting that falsey is not the same thing as None


- Why is floating-point comparison tricky?
- What values are falsey in Python?
- What do `and` and `or` return?


- Integers and floats are objects with different representation stories.
- Floats are approximate.
- Truthiness covers more than booleans.


If this notebook did its job, **Numbers, Bits, Truth, and Precision** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.


A better way to understand Numbers, Bits, Truth, and Precision is to keep moving back and forth between three views of the same code.

The first view is the human view. This is what the code is trying to say. The second view is the object view. In that view, we ask which Python objects exist, which names refer to them, and which objects are being created, reused, mutated, or discarded. The third view is the interpreter view. In that view, we ask what protocol is being used, what bytecode is being executed, and which runtime rules are controlling the result.

Many learners stop after the first view. That is enough to copy code, but it is usually not enough to explain surprising behavior. Deep understanding starts when these three views become one mental movement instead of three unrelated topics.


There is also a fourth habit that helps a lot: failure-based thinking.

For Numbers, Bits, Truth, and Precision, ask yourself what normally goes wrong when the idea is only half-understood. Does someone confuse identity with equality? Do they expect copying where only rebinding happened? Do they expect a fresh iterator when they really have an exhausted one? Do they trust a default value that is secretly shared? Do they think a class attribute is per-instance state? Once you can predict the common failure modes before they happen, your understanding becomes much more stable.


In [6]:
import dis
import inspect

def probe_runtime(value):
    local_copy = value
    frame = inspect.currentframe()
    print('locals now:', frame.f_locals)
    return local_copy

print(probe_runtime('demo'))
print('\nbytecode for probe_runtime:')
dis.dis(probe_runtime)


locals now: {'value': 'demo', 'local_copy': 'demo', 'frame': <frame at 0x000001BBD06D7DC0, file 'C:\\Users\\VIHAAN\\AppData\\Local\\Temp\\ipykernel_29940\\939914315.py', line 7, code probe_runtime>}
demo

bytecode for probe_runtime:
  4           0 RESUME                   0

  5           2 LOAD_FAST                0 (value)
              4 STORE_FAST               1 (local_copy)

  6           6 LOAD_GLOBAL              0 (inspect)
             18 LOAD_METHOD              1 (currentframe)
             40 PRECALL                  0
             44 CALL                     0
             54 STORE_FAST               2 (frame)

  7          56 LOAD_GLOBAL              5 (NULL + print)
             68 LOAD_CONST               1 ('locals now:')
             70 LOAD_FAST                2 (frame)
             72 LOAD_ATTR                3 (f_locals)
             82 PRECALL                  2
             86 CALL                     2
             96 POP_TOP

  8          98 LOAD_FAST        

A stronger number model asks not only ?what is the value?? but also ?what representation is producing this value?? The same mathematical intention can be represented by an int, float, Decimal, or Fraction, and each choice changes the tradeoff between speed, exactness, and convenience.

That is why expert answers about numeric behavior usually mention representation, not only syntax.


In [7]:
from decimal import Decimal
from fractions import Fraction
print('float repr:', repr(0.1 + 0.2))
print('decimal repr:', Decimal('0.1') + Decimal('0.2'))
print('fraction repr:', Fraction(1, 10) + Fraction(2, 10))
print('0.1 exact ratio:', (0.1).as_integer_ratio())


float repr: 0.30000000000000004
decimal repr: 0.3
fraction repr: 3/10
0.1 exact ratio: (3602879701896397, 36028797018963968)


A more mature understanding of this topic comes from accepting that machine representation is part of the meaning of code. If a value is stored as a float, you are living with binary approximation. If it is stored as Decimal, you are paying for different guarantees. If an object is being used in a conditional, then truthiness rules are in play whether or not the object was literally `True` or `False`. This is where careful Python feels more like engineering and less like memorized syntax.


## Final Takeaway

The deepest way to revise **Numbers, Bits, Truth, and Precision** is to stop treating it as a collection of syntax facts and instead treat it as a runtime story. Ask what objects are involved, what names or attributes point to those objects, what frame is active, and what protocol or bytecode path Python is following. If you can explain this topic at those four levels, then you understand much more than the visible syntax.

In interview terms, the strongest answer is usually not the shortest definition. It is a calm explanation of what Python is doing under the surface and why the behavior follows from that model. For revision, come back to this notebook and ask yourself: could I explain this entire chapter with one small code example, one memory-level explanation, and one interpreter-level explanation? If yes, the topic is becoming yours.
